In [ ]:
# Library untuk membaca dan mengolah data

import pandas as pd
from pathlib import Path

In [ ]:
# Menentukan lokasi folder project

BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Mapping nama bulan Indonesia ke angka bulan

months = {
    "Januari": "01",
    "Februari": "02",
    "Maret": "03",
    "April": "04",
    "Mei": "05",
    "Juni": "06",
    "Juli": "07",
    "Agustus": "08",
    "September": "09",
    "Oktober": "10",
    "November": "11",
    "Desember": "12",
}

In [ ]:
# Membaca data inflasi tahun 2010

file_path = (
    RAW_DIR
    / "Inflasi"
    / "Inflasi Bulanan (M-to-M), 2010.csv"
)

df = pd.read_csv(file_path)

df.head()

In [ ]:
# Membaca data inflasi tahun 2010

file_path = (
    RAW_DIR
    / "Inflasi"
    / "Inflasi Bulanan (M-to-M), 2010.csv"
)

df = pd.read_csv(file_path)

df.head()

In [ ]:
# Mengambil hanya data inflasi nasional Indonesia

indonesia_row = df[
    df.iloc[:, 0].astype(str)
    .str.contains("INDONESIA", case=False, na=False)
]

indonesia_row

In [ ]:
# Mengambil nama bulan
# Baris ke-2 berisi Januari sampai Desember

month_names = df.iloc[2, 1:13].tolist()

# Mengambil nilai inflasi Indonesia Januari sampai Desember

values = indonesia_row.iloc[0, 1:13].tolist()

print(month_names)
print(values)

In [ ]:
# Mengubah data tahun 2010 menjadi format time series
# Setiap baris merepresentasikan satu bulan

year = 2010

inflation_2010 = pd.DataFrame({
    "date": [f"{year}-{months[month]}-01" for month in month_names],
    "inflation": values
})

# Mengubah kolom date menjadi tipe datetime
inflation_2010["date"] = pd.to_datetime(inflation_2010["date"])

# Mengubah kolom inflation menjadi tipe numeric
inflation_2010["inflation"] = pd.to_numeric(
    inflation_2010["inflation"],
    errors="coerce"
)

inflation_2010

In [ ]:
# Memastikan tipe data sudah benar

inflation_2010.info()

In [ ]:
# Statistik ringkas data inflasi tahun 2010

inflation_2010.describe()

In [ ]:
# Function untuk membangun dataset inflasi nasional
# untuk satu tahun tertentu

def build_inflation_year(year):

    # Membaca file inflasi sesuai tahun
    file_path = (
        RAW_DIR
        / "Inflasi"
        / f"Inflasi Bulanan (M-to-M), {year}.csv"
    )

    df = pd.read_csv(file_path)

    # Mengambil baris INDONESIA
    indonesia_row = df[
        df.iloc[:, 0]
        .astype(str)
        .str.contains(
            "INDONESIA",
            case=False,
            na=False
        )
    ]

    # Mengambil nama bulan
    month_names = df.iloc[2, 1:13].tolist()

    # Mengambil nilai inflasi Indonesia
    values = indonesia_row.iloc[0, 1:13].tolist()

    # Mengubah menjadi format time series
    result = pd.DataFrame({
        "date": [
            f"{year}-{months[m]}-01"
            for m in month_names
        ],
        "inflation": values
    })

    # Mengubah tipe data
    result["date"] = pd.to_datetime(result["date"])

    result["inflation"] = pd.to_numeric(
        result["inflation"],
        errors="coerce"
    )

    return result

In [ ]:
build_inflation_year(2010)

In [ ]:
build_inflation_year(2015)

In [ ]:
# Menggabungkan seluruh data inflasi nasional
# dari tahun 2010 sampai 2025

inflation_all = pd.concat(
    [
        build_inflation_year(year)
        for year in range(2010, 2026)
    ],
    ignore_index=True
)

inflation_all

In [ ]:
# Mengecek ukuran dataset

inflation_all.shape

In [ ]:
# Mengecek periode data

print(
    inflation_all["date"].min()
)

print(
    inflation_all["date"].max()
)

In [ ]:
inflation_all.shape

In [ ]:
# Mengecek missing value

inflation_all.isnull().sum()

In [ ]:
# Mengecek apakah ada duplikasi tanggal

inflation_all["date"].duplicated().sum()

In [ ]:
# Menyimpan dataset hasil preprocessing

output_path = (
    PROCESSED_DIR
    / "inflation_indonesia_monthly.csv"
)

inflation_all.to_csv(
    output_path,
    index=False
)

print(f"Dataset saved: {output_path}")